In [ ]:
# === STANDART COLAB KURULUM HÜCRESİ ===
from google.colab import drive, userdata
drive.mount("/content/drive")

import os, sys
REPO_URL = "https://github.com/Oldzhay34/receipts-and-interactions.git"
BRANCH   = "b/1.7-binning"
try:
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = None
url = REPO_URL.replace("https://", f"https://{TOKEN}@") if TOKEN else REPO_URL

if not os.path.exists("/content/repo"):
    !git clone -q -b {BRANCH} {url} /content/repo
else:
    !cd /content/repo && git fetch -q && git checkout -q {BRANCH} && git pull -q

%cd /content/repo
!pip install -q -r requirements.txt && pip install -q -e .

if "/content/repo/src" not in sys.path:
    sys.path.insert(0, "/content/repo/src")

os.environ["DATA_ROOT"] = "/content/drive/MyDrive/CME4403/data"
from recsys.config import load_config
CFG = load_config()
print("commit:", os.popen("git rev-parse --short HEAD").read().strip())

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from recsys.io.artifact_store import ArtifactStore
from recsys.features.binning import (
    DomainBins,
    EqualFrequencyBins,
    EqualWidthBins,
    compare_binning_strategies,
)
from recsys.viz.plots import bar_counts

outputs_root = getattr(CFG, "outputs_dir", Path(os.environ["DATA_ROOT"]) / "outputs")
store = ArtifactStore(outputs_root)
loaded = store.load("loaded")

recipes = loaded["recipes"]
interactions = loaded["interactions"]

# Exploratory avg_rating (excluding rating == 0); model targets are fit on train-only in 1.9
rated_agg = (
    interactions.loc[interactions["rating"] > 0]
    .groupby("recipe_id")["rating"]
    .agg(avg_rating="mean", n_ratings="count")
    .reset_index()
)
recipes_rated = recipes.merge(rated_agg, left_on="id", right_on="recipe_id", how="inner")
print(f"All recipes: {len(recipes):,} | Rated recipes: {len(recipes_rated):,}")

In [ ]:
minutes_strategies = {
    "domain": DomainBins(
        column="minutes",
        edges=[0, 30, 60, np.inf],
        labels=["quick", "medium", "long"],
        output_col="minutes_bin",
    ),
    "equal_frequency": EqualFrequencyBins(
        column="minutes",
        n_bins=3,
        labels=["quick", "medium", "long"],
        output_col="minutes_bin",
    ),
}
minutes_comp = compare_binning_strategies(recipes, "minutes", minutes_strategies)
display(minutes_comp)

In [ ]:
rating_domain = DomainBins(
    column="avg_rating",
    edges=[0.0, 3.5, 4.5, 5.0],
    labels=["low", "medium", "high"],
    output_col="rating_class",
)
rating_eq_width = EqualWidthBins(
    column="avg_rating",
    n_bins=3,
    labels=["low", "medium", "high"],
    output_col="rating_class",
)
rating_eq_freq = EqualFrequencyBins(
    column="avg_rating",
    n_bins=3,
    labels=["low", "medium", "high"],
    output_col="rating_class",
)

rating_comp = compare_binning_strategies(
    recipes_rated,
    "avg_rating",
    {
        "domain_3.5_4.5": rating_domain,
        "equal_width": rating_eq_width,
        "equal_frequency": rating_eq_freq,
    },
)
display(rating_comp)

df_r_dom = rating_domain.fit_transform(recipes_rated)
df_r_freq = rating_eq_freq.fit_transform(recipes_rated)

fig_rating_dom = bar_counts(
    df_r_dom["rating_class"],
    top_n=5,
    title="avg_rating Classes — Domain Thresholds (3.5 / 4.5)",
    ylabel="Rating Class",
    horizontal=False,
)
fig_rating_freq = bar_counts(
    df_r_freq["rating_class"],
    top_n=5,
    title="avg_rating Classes — Equal Frequency",
    ylabel="Rating Class",
    horizontal=False,
)

In [ ]:
calories_strategies = {
    "equal_width": EqualWidthBins(
        column="calories",
        n_bins=3,
        labels=["low", "medium", "high"],
        output_col="calories_bin",
    ),
    "equal_frequency": EqualFrequencyBins(
        column="calories",
        n_bins=3,
        labels=["low", "medium", "high"],
        output_col="calories_bin",
    ),
}
calories_comp = compare_binning_strategies(recipes, "calories", calories_strategies)
display(calories_comp)

In [ ]:
saved = store.save(
    stage="binning",
    objects={
        "minutes_comparison": minutes_comp,
        "avg_rating_comparison": rating_comp,
        "calories_comparison": calories_comp,
        "fig_07_01_rating_domain": fig_rating_dom,
        "fig_07_02_rating_eq_freq": fig_rating_freq,
    },
    inputs=[loaded],
    params={
        "minutes_domain_edges": [0, 30, 60, "inf"],
        "avg_rating_domain_edges": [0.0, 3.5, 4.5, 5.0],
        "n_bins": 3,
    },
)

plt.close(fig_rating_dom)
plt.close(fig_rating_freq)

print("Saved binning artifact:", saved.version)
print("\n=== MINUTES ===\n", minutes_comp.to_string(index=False))
print("\n=== AVG_RATING ===\n", rating_comp.to_string(index=False))
print("\n=== CALORIES ===\n", calories_comp.to_string(index=False))